# Forecasting at Scale using ETS and ray (M5)

> Forecast the M5 dataset

In this notebook we show how to use `StatsForecast` and `ray` to forecast thounsands of time series in less than 6 minutes (M5 dataset). Also, we show that `StatsForecast` has better performance in time and accuracy compared to [`Prophet` running on a Spark cluster](http:nixtla.github.io/statsforecast/examples/Prophet_spark_m5.html) using DataBricks.

In this example, we used a ray cluster (AWS) of 11 instances of type m5.2xlarge (8 cores, 32 GB RAM).

## Installing StatsForecast Library

In [ ]:
%%capture
!pip install "statsforecast[ray]" neuralforecast s3fs pyarrow

In [ ]:
%%capture
from time import time

import pandas as pd
from neuralforecast.data.datasets.m5 import M5, M5Evaluation
from statsforecast import StatsForecast
from statsforecast.models import ETS

## Download data

The example uses the [M5 dataset](https://github.com/Mcompetitions/M5-methods/blob/master/M5-Competitors-Guide.pdf). It consists of `30,490` bottom time series. 

In [ ]:
Y_df = pd.read_parquet('s3://m5-benchmarks/data/train/target.parquet')
Y_df = Y_df.rename(columns={
    'item_id': 'unique_id', 
    'timestamp': 'ds', 
    'demand': 'y'
})
Y_df['ds'] = pd.to_datetime(Y_df['ds'])

In [ ]:
Y_df.head()

Since the M5 dataset contains intermittent time series, we add a constant to avoid problems during the training phase. Later, we will substract the constant from the forecasts.

In [ ]:
constant = 10
Y_df['y'] += constant

## Train the model

`StatsForecast` receives a list of models to fit each time series. Since we are dealing with Daily data, it would be benefitial to use 7 as seasonality. Observe that we need to pass the ray address to the `ray_address` argument.

In [ ]:
fcst = StatsForecast(
    df=Y_df, 
    models=[ETS(season_length=7, model='ZNA')], 
    freq='D', 
    #n_jobs=-1
    ray_address='ray://ADDRESS:10001'
)

In [ ]:
init = time()
Y_hat = fcst.forecast(28)
end = time()
print(f'Minutes taken by StatsForecast using: {(end - init) / 60}')

`StatsForecast` and `ray` took only 5.48 minutes to train `30,490` time series, compared to 18.23 minutes for Prophet and Spark.

We remove the constant.

In [ ]:
Y_hat['ETS'] -= constant

### Evaluating performance

The M5 competition used the weighted root mean squared scaled error. You can find details of the metric [here](https://github.com/Mcompetitions/M5-methods/blob/master/M5-Competitors-Guide.pdf).

In [ ]:
Y_hat = Y_hat.reset_index().set_index(['unique_id', 'ds']).unstack()
Y_hat = Y_hat.droplevel(0, 1).reset_index()

In [ ]:
*_, S_df = M5.load('./data')
Y_hat = S_df.merge(Y_hat, how='left', on=['unique_id'])

In [ ]:
M5Evaluation.evaluate(y_hat=Y_hat, directory='./data')

Also, `StatsForecast` is more accurate than Prophet, since the overall WMRSSE is `0.68`, against `0.77` obtained by prophet.